# 30. Thí nghiệm E2 & E3 trên VieNeu-TTS v3 Turbo (48 kHz)
- **E2:** Chạy cloning `8s clean` + chạy `preset_anchor` (giọng Việt bản ngữ) + chạy `same_lang_en` control.
- **E3:** Chạy lưới tham số `3s / 5s / 8s` $\times$ `clean / noisy_snr10db` $\times$ `denoise=True/False`.


In [ ]:
import os
from pathlib import Path
REPO_DIR = Path('/content/NLP_project') if Path('/content/NLP_project').exists() else Path.cwd()
if REPO_DIR.name == 'notebooks': os.chdir(REPO_DIR.parent)
# Dịch tập chính 120 câu bằng model MT đã chọn (mặc định vinai_v2)
!python scripts/run_stage.py --stage mt --config configs/mt/vinai_v2.yaml \
    --run-id main_v1 \
    --input-manifest outputs/stt/vctk_main_stt.jsonl \
    --output-manifest outputs/mt/vctk_main_mt.jsonl


In [ ]:
# E2: VieNeu Cloning 8s Clean + Native Anchor + Same-Language EN Control
!python scripts/run_stage.py --stage tts --config configs/tts/vieneu_v3turbo.yaml \
    --run-id main_v1 \
    --input-manifest outputs/mt/vctk_main_mt.jsonl \
    --output-manifest outputs/tts/e2_vieneu_clone8s.jsonl \
    --condition-tag clone_8s_clean_denoise_on --ref-duration 8s --ref-quality clean

!python scripts/run_stage.py --stage tts --config configs/tts/vieneu_v3turbo.yaml \
    --run-id main_v1 \
    --input-manifest outputs/mt/vctk_main_mt.jsonl \
    --output-manifest outputs/tts/e2_vieneu_anchor.jsonl \
    --condition-tag preset_native_anchor --preset-anchor

!python scripts/run_stage.py --stage tts --config configs/tts/vieneu_v3turbo.yaml \
    --run-id main_v1 \
    --input-manifest outputs/mt/vctk_main_mt.jsonl \
    --output-manifest outputs/tts/e2_vieneu_en_control.jsonl \
    --condition-tag same_lang_en_control --same-lang-en


In [ ]:
# E3: Chạy toàn bộ lưới Reference Length (3s/5s/8s) x Noise/Denoising (4 điều kiện)
!python scripts/run_stage.py --stage e3_grid --config configs/tts/vieneu_v3turbo.yaml \
    --run-id main_v1 \
    --input-manifest outputs/mt/vctk_main_mt.jsonl \
    --output-manifest outputs/tts/e3_vieneu.jsonl
